# 9주차 과제 — 내풀이 v3

In [1]:
# 실습 환경 세팅
!pip install -q langchain langchain-core langchain-openai langchain-google-genai langgraph tavily-python python-dotenv

In [2]:
# API 키 설정 (로컬 환경)
import os
from dotenv import load_dotenv
load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 필요"
print("API 키 설정 완료")

API 키 설정 완료


---
## 문제 1. LangChain 기본 모델 호출

In [3]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

# TODO 1: 모델 초기화
model = init_chat_model(model="openai:gpt-4o-mini")

# TODO 2: HumanMessage 생성 및 invoke
response = model.invoke([HumanMessage(content="LangChain이 무엇인지 한 문장으로 설명해줘.")])

# TODO 3: 응답 내용 출력
print(response.content)

LangChain은 언어 모델과 다양한 외부 데이터 소스를 결합하여 자연어 처리 애플리케이션을 구축하고 관리할 수 있도록 돕는 프레임워크입니다.


---
## 문제 2. SystemMessage + HumanMessage 대화 구성

In [4]:
from langchain_core.messages import SystemMessage, HumanMessage

# TODO 1: 메시지 리스트 구성
messages = [
    SystemMessage(content="너는 AI 에이전트 전문가야. 모든 답변을 2문장 이내로 해줘."),
    HumanMessage(content="LangGraph를 설명해줘."),
]

# TODO 2: 모델 호출
response = model.invoke(messages)

print(response.content)

LangGraph는 자연어 처리(NLP)와 그래프 데이터베이스를 결합하여 복잡한 언어 구조를 시각화하고 분석할 수 있는 도구입니다. 이를 통해 텍스트 데이터를 그래프 형태로 변환하여 관계성과 패턴을 쉽게 식별할 수 있습니다.


---
## 문제 3. Structured Output

In [5]:
from pydantic import BaseModel, Field

# TODO 1: Pydantic 스키마 정의
class FrameworkInfo(BaseModel):
    name: str = Field(description="AI 프레임워크 이름")
    description: str = Field(description="한 줄 설명")

# TODO 2: structured output 모델 구성
structured_model = model.with_structured_output(FrameworkInfo)

# TODO 3: 호출
result = structured_model.invoke("LangGraph를 소개해줘.")

print(f"name: {result.name}")
print(f"description: {result.description}")

name: LangGraph
description: LangGraph는 다양한 언어 처리 및 분석 작업을 위해 최적화된 그래프 기반 프레임워크입니다. 자연어 처리(NLP), 기계 번역, 문서 요약 등 다양한 언어 관련 작업에 활용될 수 있으며, 그래프 구조를 통해 언어의 복잡한 관계와 의미를 더 잘 이해하고 모델링할 수 있도록 돕습니다. LangGraph는 사용자가 여러 개의 데이터 소스를 통합하고, 언어 데이터의 특징을 시각화하며, 효율적으로 학습할 수 있는 환경을 제공합니다.


---
## 문제 4. LangGraph State 정의

In [6]:
import operator
from typing_extensions import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

# TODO: AgentState 정의
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    topic: str
    summary: str

# 검증
assert "messages" in AgentState.__annotations__
assert "topic" in AgentState.__annotations__
assert "summary" in AgentState.__annotations__
print("State 정의 완료:", list(AgentState.__annotations__.keys()))

State 정의 완료: ['messages', 'topic', 'summary']


---
## 문제 5. StateGraph 기본 구성

In [7]:
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END

# TODO 1: respond 노드 함수
def respond(state: AgentState):
    last_content = state["messages"][-1].content
    return {"messages": [AIMessage(content=f"{last_content} 에 대해 답변드릴게요.")]}

# TODO 2: StateGraph 구성
builder = StateGraph(AgentState)
builder.add_node("respond", respond)
builder.add_edge(START, "respond")
builder.add_edge("respond", END)

# TODO 3: 컴파일 및 실행
graph = builder.compile()
result = graph.invoke({"messages": [HumanMessage(content="LangGraph")]})
print(result["messages"][-1].content)

LangGraph 에 대해 답변드릴게요.


---
## 문제 6. 조건부 라우팅

In [8]:
from typing_extensions import Literal

# TODO 1: route 함수
def route(state: AgentState) -> Literal["long", "short"]:
    last_content = state["messages"][-1].content
    if len(last_content) >= 10:
        return "long"
    return "short"

# TODO 2: 노드 함수
def long_response(state: AgentState):
    return {"messages": [AIMessage(content="긴 질문이에요.")]}

def short_response(state: AgentState):
    return {"messages": [AIMessage(content="짧은 질문이에요.")]}

# TODO 3: 그래프 구성 (add_conditional_edges 사용)
builder = StateGraph(AgentState)
builder.add_node("long", long_response)
builder.add_node("short", short_response)
builder.add_conditional_edges(
    START,
    route,
    {"long": "long", "short": "short"},
)
builder.add_edge("long", END)
builder.add_edge("short", END)

graph = builder.compile()

# TODO 4: 실행
result_long = graph.invoke({"messages": [HumanMessage(content="LangGraph에 대해 자세히 설명해줘")]})
result_short = graph.invoke({"messages": [HumanMessage(content="안녕")]})
print(result_long["messages"][-1].content)
print(result_short["messages"][-1].content)

긴 질문이에요.
짧은 질문이에요.


---
## 문제 7. Tool 정의 및 tool_node 구현

In [9]:
from langchain_core.tools import tool, InjectedToolArg
from langchain_core.messages import ToolMessage, AIMessage
from typing_extensions import Annotated

# TODO 1: word_count_tool 정의
@tool(parse_docstring=True)
def word_count_tool(
    text: str,
    max_words: Annotated[int, InjectedToolArg] = 100,
) -> str:
    """텍스트의 단어 수를 반환한다.

    Args:
        text: 단어 수를 셀 텍스트
    """
    count = len(text.split())
    return str(min(count, max_words))

# TODO 2: tools_by_name 구성
tools = [word_count_tool]
tools_by_name = {t.name: t for t in tools}

# TODO 3: tool_node 구현
def tool_node(state):
    tool_calls = state["messages"][-1].tool_calls
    outputs = []
    for tc in tool_calls:
        result = tools_by_name[tc["name"]].invoke(tc["args"])
        outputs.append(ToolMessage(content=str(result), name=tc["name"], tool_call_id=tc["id"]))
    return {"messages": outputs}

# 검증
fake_ai_msg = AIMessage(
    content="",
    tool_calls=[{"name": "word_count_tool", "args": {"text": "Hello LangGraph"}, "id": "tc1", "type": "tool_call"}]
)
test_result = tool_node({"messages": [fake_ai_msg]})
assert len(test_result["messages"]) == 1
assert isinstance(test_result["messages"][0], ToolMessage)
print("tool_node 테스트 통과")
print("단어 수 결과:", test_result["messages"][0].content)

tool_node 테스트 통과
단어 수 결과: 2


---
## 문제 8. output_schema 적용 및 실행 검증

In [10]:
# TODO 1: ResearchState 정의
class ResearchState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    result: str

# TODO 2: ResearchOutputState 정의
class ResearchOutputState(TypedDict):
    result: str

# TODO 3: llm_call 노드
def llm_call(state: ResearchState):
    response = model.invoke(state["messages"])
    return {"messages": [response]}

# TODO 4: finalize 노드
def finalize(state: ResearchState):
    last_content = state["messages"][-1].content
    return {"result": last_content}

# TODO 5: StateGraph 구성 (output_schema 적용)
research_builder = StateGraph(ResearchState, output_schema=ResearchOutputState)
research_builder.add_node("llm_call", llm_call)
research_builder.add_node("finalize", finalize)
research_builder.add_edge(START, "llm_call")
research_builder.add_edge("llm_call", "finalize")
research_builder.add_edge("finalize", END)

research_graph = research_builder.compile()

# TODO 6: 실행 및 검증
result = research_graph.invoke({"messages": [HumanMessage(content="LangGraph를 한 문장으로 설명해줘.")]})

print("result keys:", result.keys())
assert set(result.keys()) == {"result"}, f"output_schema가 올바르지 않습니다: {result.keys()}"
print("output_schema 검증 통과")
print("result:", result["result"])

result keys: dict_keys(['result'])
output_schema 검증 통과
result: LangGraph는 다양한 언어 처리 작업을 수행할 수 있도록 설계된 그래프 기반의 언어 모델입니다.


---
## 문제 9. ReAct 에이전트 전체 구현

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from typing_extensions import TypedDict, Annotated, Sequence, Literal
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from tavily import TavilyClient

tavily_client = TavilyClient()

# TODO 1: simple_search tool 정의
@tool(parse_docstring=True)
def simple_search(query: str) -> str:
    """웹을 검색하고 결과를 반환한다.

    Args:
        query: 검색 쿼리
    """
    result = tavily_client.search(query=query, max_results=2)
    return "\n".join([r["content"] for r in result["results"]])

# TODO 2: ReactState 정의
class ReactState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

# TODO 3: 모델 + tool 바인딩
react_model = init_chat_model(model="openai:gpt-4o-mini")
react_model_with_tools = react_model.bind_tools([simple_search])

# TODO 4: llm_call 노드
def react_llm_call(state: ReactState):
    response = react_model_with_tools.invoke(
        [SystemMessage(content="너는 웹 검색 에이전트야.")] + list(state["messages"])
    )
    return {"messages": [response]}

# TODO 5: tool_node 구현
react_tools_by_name = {simple_search.name: simple_search}

def react_tool_node(state: ReactState):
    tool_calls = state["messages"][-1].tool_calls
    outputs = []
    for tc in tool_calls:
        obs = react_tools_by_name[tc["name"]].invoke(tc["args"])
        outputs.append(ToolMessage(content=str(obs), name=tc["name"], tool_call_id=tc["id"]))
    return {"messages": outputs}

# TODO 6: should_continue 라우팅 함수
def should_continue(state: ReactState) -> Literal["tool_node", "__end__"]:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tool_node"
    return "__end__"

# TODO 7: 그래프 구성
react_builder = StateGraph(ReactState)
react_builder.add_node("llm_call", react_llm_call)
react_builder.add_node("tool_node", react_tool_node)
react_builder.add_edge(START, "llm_call")
react_builder.add_conditional_edges(
    "llm_call",
    should_continue,
    {"tool_node": "tool_node", "__end__": END},
)
react_builder.add_edge("tool_node", "llm_call")

react_graph = react_builder.compile()

# TODO 8: 실행
result = react_graph.invoke(
    {"messages": [HumanMessage(content="2026년 LangChain의 최신 버전을 검색해줘.")]}
)
print(result["messages"][-1].content)

---
## 문제 10. 멀티턴 대화 + compress 노드 구현

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from typing_extensions import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage, HumanMessage, filter_messages
from langchain.chat_models import init_chat_model
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

# TODO 1: MultiTurnState 정의
class MultiTurnState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    compressed_result: str

# TODO 2: research_node
def research_node(state: MultiTurnState):
    research_model = init_chat_model(model="openai:gpt-4o-mini")
    response = research_model.invoke(state["messages"])
    return {"messages": [response]}

# TODO 3: compress_node (filter_messages 사용)
compress_model = init_chat_model(model="google_genai:gemini-2.5-flash")

def compress_node(state: MultiTurnState):
    ai_messages = filter_messages(state["messages"], include_types=["ai"])
    combined = "\n".join([m.content for m in ai_messages])
    response = compress_model.invoke(
        [HumanMessage(content=f"다음 내용을 한 문장으로 요약해줘:\n{combined}")]
    )
    return {"compressed_result": response.content}

# TODO 4: 그래프 구성 + checkpointer 적용
checkpointer = InMemorySaver()

mt_builder = StateGraph(MultiTurnState)
mt_builder.add_node("research_node", research_node)
mt_builder.add_node("compress_node", compress_node)
mt_builder.add_edge(START, "research_node")
mt_builder.add_edge("research_node", "compress_node")
mt_builder.add_edge("compress_node", END)

mt_graph = mt_builder.compile(checkpointer=checkpointer)

# TODO 5: 멀티턴 실행
thread = {"configurable": {"thread_id": "turn-test"}}

result1 = mt_graph.invoke(
    {"messages": [HumanMessage(content="LangGraph의 주요 특징을 알려줘.")]},
    config=thread,
)
print(f"Turn 1 messages: {len(result1['messages'])}개")

result2 = mt_graph.invoke(
    {"messages": [HumanMessage(content="그 중 가장 중요한 것은 뭐야?")]},
    config=thread,
)
print(f"Turn 2 messages: {len(result2['messages'])}개")

# TODO 6: 검증
assert len(result2["messages"]) > len(result1["messages"]), \
    "멀티턴 대화에서 messages가 누적되지 않았습니다."
print("멀티턴 누적 확인")
print("Turn 2 compressed:", result2["compressed_result"])